# Image creation from audio data



# Image generation process
- Compute dB scaled mel power spectrum over 5 seconds interval.
- Use primary label for each of these intervals.
- Pad to 5 second images if we have a minimal duration.
- Consider a maximum duration for a maximum number of images created per file.
- Create images independently.

In [1]:
import os, pathlib
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from pydantic import BaseModel as ConfigBaseModel
from sklearn.preprocessing import LabelEncoder
from joblib import delayed, Parallel
import librosa
print("librosa:", librosa.__version__)
import tensorflow as tf
print("tensorflow:", tf.__version__)
import cv2
print("opencv:", cv2.__version__)
from IPython.display import Audio
from joblib import Parallel, delayed

librosa: 0.10.2.post1
tensorflow: 2.18.0
opencv: 4.10.0


# Config

In [2]:
from pydantic import BaseModel
from typing import Tuple

class Config(BaseModel):
    # Datos de configuración principales
    base_dir: str = "./Users/camcortes/Documents/birds-sounds/"
    train_sound_dir: str = "./Users/camcortes/Documents/birds-sounds/songs/"
    path_train: str = "data/val_muestra.csv"
    sample_rate: int = 32_000
    # Configuración del espectrograma
    img_size: Tuple[int, int] = (128, 256)
    seconds: int = 5
    num_offset_max: int = 24
    min_duration: float = 0.5
    n_fft: int = 2048
    n_mels: int = 128  # Corresponde a img_size[0]
    # Cálculo predefinido de hop_length
    hop_length: int = 622  # Calculado como (5 * 32000 - 2048) // (256 - 1)
    center: bool = False
    fmin: int = 500
    fmax: int = 12_500
    top_db: int = 80

    # Configuración de salida
    out_dir: str = "./images_spectograms/val/"
    jpeg_quality: int = 100

# Crear una instancia de la configuración
cfg = Config()
cfg.dict()

/var/folders/_1/9ctwy_pn6bv2hyw11l_gqhm9855lbl/T/ipykernel_30015/1548527771.py:33: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  cfg.dict()


{'base_dir': './Users/camcortes/Documents/birds-sounds/',
 'train_sound_dir': './Users/camcortes/Documents/birds-sounds/songs/',
 'path_train': 'data/val_muestra.csv',
 'sample_rate': 32000,
 'img_size': (128, 256),
 'seconds': 5,
 'num_offset_max': 24,
 'min_duration': 0.5,
 'n_fft': 2048,
 'n_mels': 128,
 'hop_length': 622,
 'center': False,
 'fmin': 500,
 'fmax': 12500,
 'top_db': 80,
 'out_dir': './images_spectograms/val/',
 'jpeg_quality': 100}

# Prepare

In [3]:
data = pd.read_csv(cfg.path_train)
data["path_ogg"] = data["audio_path"]

In [4]:
le = LabelEncoder()
data['label_encoder'] = le.fit_transform(data['label'])

In [5]:
def get_duration(rec):
    return librosa.get_duration(path=rec["path_ogg"])

def get_duration_df(df):
    return df.apply(get_duration, axis=1)

In [6]:
durations = Parallel(n_jobs=os.cpu_count(), verbose=1)(
    delayed(get_duration_df)(sub) 
    for sub in np.array_split(data, os.cpu_count())
)
data["duration"] = pd.concat(durations)

/Users/camcortes/Library/Caches/pypoetry/virtualenvs/birds-sounds-images-gIa9Kkaf-py3.10/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    3.4s remaining:   13.5s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    3.6s finished


In [7]:
data.head(10)

,label,Genus,Family,audio_path,Suborder,path_ogg,label_encoder,duration
0,Pheugopedius rutilus,Pheugopedius,Troglodytidae,./songs/Troglodytidae/Pheugopedius/Pheugopediu...,passeri,./songs/Troglodytidae/Pheugopedius/Pheugopediu...,28,147.764331
1,Cistothorus platensis,Cistothorus,Troglodytidae,./songs/Troglodytidae/Cistothorus/Cistothorus ...,passeri,./songs/Troglodytidae/Cistothorus/Cistothorus ...,12,54.401247
2,Campylorhynchus griseus,Campylorhynchus,Troglodytidae,./songs/Troglodytidae/Campylorhynchus/Campylor...,passeri,./songs/Troglodytidae/Campylorhynchus/Campylor...,1,27.570794
3,Troglodytes aedon,Troglodytes,Troglodytidae,./songs/Troglodytidae/Troglodytes/Troglodytes ...,passeri,./songs/Troglodytidae/Troglodytes/Troglodytes ...,33,24.392472
4,Cantorchilus nigricapillus,Cantorchilus,Troglodytidae,./songs/Troglodytidae/Cantorchilus/Cantorchilu...,passeri,./songs/Troglodytidae/Cantorchilus/Cantorchilu...,8,15.688821
5,Troglodytes aedon,Troglodytes,Troglodytidae,./songs/Troglodytidae/Troglodytes/Troglodytes ...,passeri,./songs/Troglodytidae/Troglodytes/Troglodytes ...,33,29.634535
6,Troglodytes aedon,Troglodytes,Troglodytidae,./songs/Troglodytidae/Troglodytes/Troglodytes ...,passeri,./songs/Troglodytidae/Troglodytes/Troglodytes ...,33,45.792132
7,Microcerculus marginatus,Microcerculus,Troglodytidae,./songs/Troglodytidae/Microcerculus/Microcercu...,passeri,./songs/Troglodytidae/Microcerculus/Microcercu...,21,28.998562
8,Troglodytes aedon,Troglodytes,Troglodytidae,./songs/Troglodytidae/Troglodytes/Troglodytes ...,passeri,./songs/Troglodytidae/Troglodytes/Troglodytes ...,33,8.411859
9,Pheugopedius spadix,Pheugopedius,Troglodytidae,./songs/Troglodytidae/Pheugopedius/Pheugopediu...,passeri,./songs/Troglodytidae/Pheugopedius/Pheugopediu...,30,74.978027


In [8]:
data["num_offset"] = (1 + (data["duration"] - cfg.min_duration) // cfg.seconds).astype('int')
data["num_offset"] = data["num_offset"].clip(upper=cfg.num_offset_max)

In [9]:
data.head(10)

,label,Genus,Family,audio_path,Suborder,path_ogg,label_encoder,duration,num_offset
0,Pheugopedius rutilus,Pheugopedius,Troglodytidae,./songs/Troglodytidae/Pheugopedius/Pheugopediu...,passeri,./songs/Troglodytidae/Pheugopedius/Pheugopediu...,28,147.764331,24
1,Cistothorus platensis,Cistothorus,Troglodytidae,./songs/Troglodytidae/Cistothorus/Cistothorus ...,passeri,./songs/Troglodytidae/Cistothorus/Cistothorus ...,12,54.401247,11
2,Campylorhynchus griseus,Campylorhynchus,Troglodytidae,./songs/Troglodytidae/Campylorhynchus/Campylor...,passeri,./songs/Troglodytidae/Campylorhynchus/Campylor...,1,27.570794,6
3,Troglodytes aedon,Troglodytes,Troglodytidae,./songs/Troglodytidae/Troglodytes/Troglodytes ...,passeri,./songs/Troglodytidae/Troglodytes/Troglodytes ...,33,24.392472,5
4,Cantorchilus nigricapillus,Cantorchilus,Troglodytidae,./songs/Troglodytidae/Cantorchilus/Cantorchilu...,passeri,./songs/Troglodytidae/Cantorchilus/Cantorchilu...,8,15.688821,4
5,Troglodytes aedon,Troglodytes,Troglodytidae,./songs/Troglodytidae/Troglodytes/Troglodytes ...,passeri,./songs/Troglodytidae/Troglodytes/Troglodytes ...,33,29.634535,6
6,Troglodytes aedon,Troglodytes,Troglodytidae,./songs/Troglodytidae/Troglodytes/Troglodytes ...,passeri,./songs/Troglodytidae/Troglodytes/Troglodytes ...,33,45.792132,10
7,Microcerculus marginatus,Microcerculus,Troglodytidae,./songs/Troglodytidae/Microcerculus/Microcercu...,passeri,./songs/Troglodytidae/Microcerculus/Microcercu...,21,28.998562,6
8,Troglodytes aedon,Troglodytes,Troglodytidae,./songs/Troglodytidae/Troglodytes/Troglodytes ...,passeri,./songs/Troglodytidae/Troglodytes/Troglodytes ...,33,8.411859,2
9,Pheugopedius spadix,Pheugopedius,Troglodytidae,./songs/Troglodytidae/Pheugopedius/Pheugopediu...,passeri,./songs/Troglodytidae/Pheugopedius/Pheugopediu...,30,74.978027,15


## Get spectogram image
In short we like to use 5 second interval spectograms as input images. But what should be done with the corner cases?
- There is a maximum number of offset considered for very long audio files.
- Very short files should be padded with zero to get a minimal length.

In [10]:
def get_mel_spec_db(path_ogg, offset):
    """Get dB scaled mel power spectrum"""
    required_len = cfg.seconds * cfg.sample_rate
    sig, dr = librosa.load(path=path_ogg, sr=cfg.sample_rate, offset=(offset * cfg.seconds), duration=cfg.seconds)
    sig = np.concatenate([sig, np.zeros((required_len - len(sig)), dtype=sig.dtype)])
    mel_spec = librosa.feature.melspectrogram(
        y=sig, 
        hop_length=cfg.hop_length,
        sr=cfg.sample_rate, 
        n_fft=cfg.n_fft, 
        n_mels=cfg.n_mels,
        center=cfg.center,
        fmin=cfg.fmin,
        fmax=cfg.fmax,
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max, top_db=cfg.top_db)
    return mel_spec_db

In [11]:
def normalize_img(img):
    """Normalize to uint8 image range"""
    assert img.ndim == 2, "unexpected dimension"
    v_min, v_max = np.min(img), np.max(img)
    return ((img - v_min) / (v_max - v_min) * 255).astype('uint8')

In [12]:
def process_record(rec):
    """Process a single record"""
    rec_dir = os.path.join(cfg.out_dir, rec.label)
    os.makedirs(rec_dir, exist_ok=True)
    stats = []
    base_stat = {"label": rec.label, "orig_filename": rec.audio_path}
    
    if not os.path.exists(rec.path_ogg):
        raise FileNotFoundError(f"File not found: {rec.path_ogg}")
    
    for offset in range(rec.num_offset):
        try:
            mel_spec_db = get_mel_spec_db(rec.path_ogg, offset=offset)
            img = normalize_img(mel_spec_db)
            fname = f"{pathlib.Path(rec.audio_path).stem}_{offset}.jpeg"
            path_img = os.path.join(rec_dir, fname)
            ret = cv2.imwrite(path_img, img, [cv2.IMWRITE_JPEG_QUALITY, cfg.jpeg_quality])
            stat = base_stat.copy()
            stat.update({
                "offset": offset,
                "ret": ret,
                "filename": "/".join(pathlib.Path(path_img).parts[-2:]),
            })
            stats.append(stat)
        except Exception as e:
            print(f"Error processing offset {offset} for {rec.audio_path}: {e}")
            continue
    return pd.DataFrame(stats)

def process_data(data):
    """Process dataframe"""
    errors = []
    l_stats = []
    for rec in data.itertuples():
        try:
            stats = process_record(rec)
            l_stats.append(stats)
        except FileNotFoundError as e:
            print(e)
            errors.append((rec.audio_path, str(e)))
        except Exception as e:
            print(f"Error reading {rec.audio_path}: {e}")
            errors.append((rec.audio_path, str(e)))
        gc.collect()
    combined_stats = pd.concat(l_stats, ignore_index=True)
    return combined_stats, errors

#### Dev

# Run all

In [13]:
results = Parallel(n_jobs=os.cpu_count(), verbose=1)(
    delayed(process_data)(sub) for sub in np.array_split(data, os.cpu_count())
    )

[src/libmpg123/layer3.c:INT123_do_layer3():1801] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1801] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1801] error: dequantization failed!
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:   28.9s remaining:  1.9min
[src/libmpg123/layer3.c:INT123_do_layer3():1801] error: dequantization failed!
/var/folders/_1/9ctwy_pn6bv2hyw11l_gqhm9855lbl/T/ipykernel_30015/2494747069.py:5: RuntimeWarning: invalid value encountered in divide
/var/folders/_1/9ctwy_pn6bv2hyw11l_gqhm9855lbl/T/ipykernel_30015/2494747069.py:5: RuntimeWarning: invalid value encountered in cast
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:   30.2s finished


In [14]:
img_stats = [x for r in results for x in r[0] if isinstance(x, pd.DataFrame)]

if len(img_stats):
    img_stats = pd.concat(img_stats).reset_index(drop=True)
img_stats

[]

In [15]:
print("Expected number of images:", data["num_offset"].sum())

Expected number of images: 9312


In [16]:
def convert_bytes(num):
    for x in ['bytes', 'KB', 'MB', 'GB', 'TB']:
        if num < 1024.0:
            return "%3.1f %s" % (num, x)
        num /= 1024.0

        
bs = sum(os.stat(f).st_size for f in pathlib.Path(cfg.out_dir).glob("*/*"))
print(cfg.out_dir, convert_bytes(bs))

./images_spectograms/val/ 226.6 MB
